In [37]:
import os
import subprocess
import glob
import shutil  # Add this import for cross-device moves

def get_interaction_energy(summary_file):
    """Extract IntraclashesGroup2 from FoldX summary file."""
    try:
        with open(summary_file, 'r') as f:
            for line in f:
                line = line.strip()
                # Look for data lines starting with "./"
                if line.startswith("./"):
                    parts = line.split()
                    # Column layout:
                    # 0: Pdb, 1: Group1, 2: Group2, 3: IntraclashesGroup1, 
                    # 4: IntraclashesGroup2, 5: Interaction Energy, 
                    # 6: StabilityGroup1, 7: StabilityGroup2
                    return float(parts[5])  # 5th column: IntraclashesGroup2
    except (ValueError, IndexError) as e:
        print(f"Error parsing {summary_file}: {e}")
    except Exception as e:
        print(f"Error reading {summary_file}: {e}")
    return None

# Define input and output folders
input_folder = "/media/emel/WD_3tb/ab-ag_complex_features/module/Ab-Ag_complex_features/examples/pdbs"
output_folder = "/media/emel/d/slim_af2.3/NEW/foldx_results/"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get all PDB files from input folder
pdb_files = glob.glob(os.path.join(input_folder, "*.pdb"))

if not pdb_files:
    print(f"No PDB files found in {input_folder}")
else:
    print(f"Found {len(pdb_files)} PDB files")

# Store results
results = []

for pdb_path in pdb_files:
    # Get just the filename
    pdb_file = os.path.basename(pdb_path)
    model_id = pdb_file.replace('.pdb', '')
    
    # print(f"Processing: {pdb_file}")
    
    # Change to input directory for FoldX to find the PDB
    os.chdir(input_folder)
    
    cmd = [
        "foldx_20251231",
        "--command=AnalyseComplex",
        f"--pdb={pdb_file}",
        "--analyseComplexChains=A,B"
    ]
    
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        
        # Look for summary file in current directory
        summary_file = f"Summary_{model_id}_AC.fxout"
        
        if os.path.exists(summary_file):
            energy = get_interaction_energy(summary_file)
            # print(f"✓ Success: {pdb_file} → Energy: {energy}")
            
            # Move ONLY Summary files to output folder
            for out_file in glob.glob(f"Summary_{model_id}*.fxout"):
                dest_path = os.path.join(output_folder, out_file)
                try:
                    shutil.move(out_file, dest_path)  # Use shutil.move instead of os.rename
                except Exception as e:
                    print(f"Warning: Could not move {out_file}: {e}")
        else:
            energy = None
            print(f"⚠ Warning: Summary file not found for {pdb_file}")
            
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed: {pdb_file}")
        print(e.stderr)
        energy = None
    
    results.append({
        'model_id': model_id,
        'pdb_file': pdb_file,
        'interaction_energy': energy
    })

# Save results to CSV
import pandas as pd
df_results = pd.DataFrame(results)
df_results.to_csv('/media/emel/d/slim_af2.3/NEW/fold_results.csv', index=False)

Found 35105 PDB files


In [2]:
import os
import subprocess
import glob

# Define folders
predicted_folder = "/media/emel/WD_3tb/ab-ag_complex_features/module/Ab-Ag_complex_features/examples/pdbs"
native_folder = "/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/original_pdbs/"  # Original native PDBs
output_folder = "/media/emel/d/slim_af2.3/NEW/tmscores/"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get all predicted PDB files
predicted_pdbs = glob.glob(os.path.join(predicted_folder, "**/*.pdb"), recursive=True)

total_files = len(predicted_pdbs)
print(f"Found {total_files} predicted PDB files")

results = []
success_count = 0
failed_count = 0

for idx, pred_pdb in enumerate(predicted_pdbs, 1):
    # Extract PDB ID from filename
    filename = os.path.basename(pred_pdb)
    pdb_code = filename[:7]
    
    # Find corresponding native PDB
    native_pdb = os.path.join(native_folder, f"{pdb_code}.pdb")
    
    if not os.path.exists(native_pdb):
        failed_count += 1
        results.append({
            'predicted_pdb': filename,
            'native_pdb': f"{pdb_code}.pdb",
            'output_file': 'N/A',
            'status': 'native_not_found'
        })
        continue
    
    # Create output filename
    output_name = filename
    output_file = os.path.join(output_folder, f"{output_name}.tm")
    
    # Run TMscore
    cmd = [
        "TMscore",
        "-c",
        pred_pdb,
        native_pdb,
        "-seq"
    ]
    
    try:
        # Run command and capture output
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        
        # Save output to file
        with open(output_file, 'w') as f:
            f.write(result.stdout)
        
        success_count += 1
        results.append({
            'predicted_pdb': filename,
            'native_pdb': f"{pdb_code}.pdb",
            'output_file': output_file,
            'status': 'success'
        })
        
    except subprocess.CalledProcessError as e:
        failed_count += 1
        results.append({
            'predicted_pdb': filename,
            'native_pdb': f"{pdb_code}.pdb",
            'output_file': output_file,
            'status': 'failed'
        })
    
    # Print progress every 100 files
    if idx % 5000 == 0 or idx == total_files:
        print(f"Progress: {idx}/{total_files} | Success: {success_count} | Failed: {failed_count}")

print(f"\n{'='*50}")
print(f"COMPLETED")
print(f"{'='*50}")
print(f"Total processed: {len(results)}")
print(f"Successful: {success_count}")
print(f"Failed: {failed_count}")

Found 35105 predicted PDB files
Progress: 5000/35105 | Success: 5000 | Failed: 0
Progress: 10000/35105 | Success: 10000 | Failed: 0
Progress: 15000/35105 | Success: 15000 | Failed: 0
Progress: 20000/35105 | Success: 20000 | Failed: 0
Progress: 25000/35105 | Success: 25000 | Failed: 0
Progress: 30000/35105 | Success: 30000 | Failed: 0
Progress: 35000/35105 | Success: 35000 | Failed: 0
Progress: 35105/35105 | Success: 35105 | Failed: 0

COMPLETED
Total processed: 35105
Successful: 35105
Failed: 0


In [4]:
import os
import glob
import pandas as pd
import re

def parse_tmscore_file(tm_file):
    """Extract TM-score, MaxSub-score, GDT-TS-score, and GDT-HA-score from TMscore output."""
    scores = {
        'tm_score': None,
        'maxsub_score': None,
        'gdt_ts_score': None,
        'gdt_ha_score': None
    }
    
    try:
        with open(tm_file, 'r') as f:
            content = f.read()
            
            # Extract TM-score
            tm_match = re.search(r'TM-score\s*=\s*([\d.]+)', content)
            if tm_match:
                scores['tm_score'] = float(tm_match.group(1))
            
            # Extract MaxSub-score
            maxsub_match = re.search(r'MaxSub-score=\s*([\d.]+)', content)
            if maxsub_match:
                scores['maxsub_score'] = float(maxsub_match.group(1))
            
            # Extract GDT-TS-score
            gdt_ts_match = re.search(r'GDT-TS-score=\s*([\d.]+)', content)
            if gdt_ts_match:
                scores['gdt_ts_score'] = float(gdt_ts_match.group(1))
            
            # Extract GDT-HA-score
            gdt_ha_match = re.search(r'GDT-HA-score=\s*([\d.]+)', content)
            if gdt_ha_match:
                scores['gdt_ha_score'] = float(gdt_ha_match.group(1))
                
    except Exception as e:
        print(f"Error parsing {tm_file}: {e}")
    
    return scores

# Define folder containing TMscore output files
tmscore_folder = "/media/emel/d/slim_af2.3/NEW/tmscores/"

# Get all .tm files
tm_files = glob.glob(os.path.join(tmscore_folder, "*.tm"))

print(f"Found {len(tm_files)} TMscore output files")

# Parse all files
data = []
for tm_file in tm_files:
    filename = os.path.basename(tm_file)
    pdb_name = filename.replace('.tm', '')
    
    scores = parse_tmscore_file(tm_file)
    
    data.append({
        'model_id': pdb_name,
        'tm_score': scores['tm_score'],
        'maxsub_score': scores['maxsub_score'],
        'gdt_ts_score': scores['gdt_ts_score'],
        'gdt_ha_score': scores['gdt_ha_score']
    })

# Create DataFrame
df = pd.DataFrame(data)

# Display first few rows
print(f"\nExtracted scores from {len(df)} files:")
df.to_csv('/media/emel/d/slim_af2.3/NEW/tmscore_results.csv', index=False)
# print(f"Results saved to: {output_csv}")

# Show any files with missing scores
missing_scores = df[df[['tm_score', 'maxsub_score', 'gdt_ts_score', 'gdt_ha_score']].isnull().any(axis=1)]
if len(missing_scores) > 0:
    print(f"\n⚠ Warning: {len(missing_scores)} files have missing scores")
    print(missing_scores[['pdb_name']].head())

Found 35105 TMscore output files

Extracted scores from 35105 files:
